## 1. Import Libraries & modules

In [1]:
import joblib

# Scikit-learn modules
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import class_weight

# Data processing libraries
import numpy as np
import pandas as pd
import preprocess           # Preprocess.py 

# Plotting libraries
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# 3 different algorithms
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from lightgbm import LGBMClassifier

## 2. Import & Explore Data

In [2]:
df = pd.read_csv("data/malicious_url.csv")
extracted_data = df['url'].apply(preprocess.extract_all_features).tolist()
# Concat trực tiếp: Bảng gốc + Bảng features

print("Dataframe shape: ", df.shape, "\n")
print("Types or url: ", df.type.value_counts())

df.head()

Dataframe shape:  (651191, 2) 

Types or url:  type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64


,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement


**Feature Engineering**

In [3]:
df['type'].value_counts()

type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64

### Encoding the Target

In [4]:
X = pd.DataFrame(extracted_data)
X.columns = X.columns.astype(str) # Chống lỗi hiển thị 'f'

le = LabelEncoder()
Y = le.fit_transform(df['type'])

feature_names = X.columns.tolist()

**Train Test Split**

In [5]:
weights = class_weight.compute_sample_weight(class_weight='balanced', y=Y)

# 5. Split Data
x_train, x_test, y_train, y_test, w_train, w_test = train_test_split(
    X, Y, weights, test_size=0.2, stratify=Y, random_state=5
)

# Model Bulding

## 1. XGBoost Classification

In [6]:
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.05,
    reg_lambda=1,
    random_state=42
)

model.fit(x_train, y_train, sample_weight=w_train)

# 6. Lưu mọi thứ
joblib.dump(model, "xgb_model.pkl")
joblib.dump(le, "label_encoder.pkl")
joblib.dump(feature_names, "feature_names.pkl") 


['feature_names.pkl']

In [7]:
rf_model = RandomForestClassifier(
    n_estimators=100,      # Số lượng cây (thường 100 là chuẩn)
    max_depth=15,        # RF thường để cây mọc tự do
    class_weight='balanced',
    random_state=42,
    n_jobs=-1              # Dùng toàn bộ nhân CPU để train cho nhanh
)

rf_model.fit(x_train, y_train)

# 6. Kiểm tra kết quả
y_pred = rf_model.predict(x_test)
print("\n--- KẾT QUẢ RANDOM FOREST ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# 7. Lưu model
joblib.dump(rf_model, "rf_model.pkl")
print("Đã lưu rf_model.pkl")


--- KẾT QUẢ RANDOM FOREST ---
Accuracy: 87.25%
              precision    recall  f1-score   support

      benign       0.97      0.86      0.91     85621
  defacement       0.72      0.94      0.82     19292
     malware       0.92      0.91      0.92      6504
    phishing       0.69      0.83      0.75     18822

    accuracy                           0.87    130239
   macro avg       0.83      0.89      0.85    130239
weighted avg       0.89      0.87      0.88    130239

Đã lưu rf_model.pkl


In [8]:
from sklearn.cluster import KMeans
import joblib
import pandas as pd

# 1. Load data đã trích xuất features (X)
# Giả sử X là DataFrame chứa 14 đặc trưng của ông
print("Đang huấn luyện K-Means để phân cụm dữ liệu...")

# 2. Khởi tạo K-Means
# Ta chọn n_clusters=4 vì mình có 4 nhãn (Benign, Phishing, Malware, Defacement)
# Mục tiêu xem máy tự chia có khớp với thực tế không
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans.fit(X) 

# 3. Lưu model K-Means
joblib.dump(kmeans, "kmeans_model.pkl")
print("Đã lưu kmeans_model.pkl!")

# 4. Kiểm tra thử một chút (Optional)
clusters = kmeans.labels_
df_check = pd.DataFrame({'Actual_Type': df['type'], 'Cluster': clusters})
print("\n--- Bảng đối chiếu nhanh (Gợi ý) ---")
print(pd.crosstab(df_check['Actual_Type'], df_check['Cluster']))

Đang huấn luyện K-Means để phân cụm dữ liệu...
Đã lưu kmeans_model.pkl!

--- Bảng đối chiếu nhanh (Gợi ý) ---
Cluster           0     1      2       3
Actual_Type                             
benign       271607  6526  31444  118526
defacement    25774  2039  21946   46698
malware       18644    71   1114   12691
phishing      76141  1960   3903   12107
